In [7]:
import json
import pyomo.environ as pyo
import numpy as np

In [8]:
with open("../../data/expected_returns_covariance_matrix.json") as f:
    data = json.load(f)
    tickers = data["tickers"] #list
    exp_ret = data["expected_returns"] #list

with open("../../data/scenario_cumulative_returns.json") as f:
    scenarios = json.load(f) #nested dict 
    scenarios['0']['TSM'] # example of how to access an entry

In [9]:
scenarios['0']['TSM']

1.3833857691851517

In [10]:
help(pyo.Var)

Help on class Var in module pyomo.core.base.var:

class Var(pyomo.core.base.indexed_component.IndexedComponent, pyomo.core.base.indexed_component.IndexedComponent_NDArrayMixin)
 |  Var(*args, **kwargs)
 |
 |  A numeric variable, which may be defined over an index.
 |
 |  Args:
 |      domain (Set or function, optional): A Set that defines valid
 |          values for the variable (e.g., ``Reals``, ``NonNegativeReals``,
 |          ``Binary``), or a rule that returns Sets.  Defaults to ``Reals``.
 |      within (Set or function, optional): An alias for ``domain``.
 |      bounds (tuple or function, optional): A tuple of ``(lower, upper)``
 |          bounds for the variable, or a rule that returns tuples.
 |          Defaults to ``(None, None)``.
 |      initialize (float or function, optional): The initial value for
 |          the variable, or a rule that returns initial values.
 |      rule (float or function, optional): An alias for ``initialize``.
 |      dense (bool, optional): In

In [11]:
def solve_cvar(alpha, scenarios, tickers, expected_returns, target_return, 
               tolerance=0.0001, time_limit=10, solver_options=None):
    """
    Minimize Conditional Value-at-Risk (CVaR) of portfolio returns.
    
    Args:
        alpha: CVaR confidence level (e.g., 0.95 for 95% CVaR)
        scenarios: Dict of scenarios with cumulative returns, keyed by scenario ID
                   e.g., {'0': {'TICKER': multiplier, ...}, '1': {...}, ...}
        tickers: List of ticker symbols
        expected_returns: List of expected returns (geometric multipliers, e.g., 1.02)
        target_return: Target portfolio expected return (geometric multiplier)
        tolerance: Tolerance band around target return (default 0.0001)
        time_limit: Solver time limit in seconds (default 10)
        solver_options: Optional dict of additional solver options
    
    Returns:
        Tuple: (weights_dict, gamma_value, cvar_objective, solver_result)
    """
    S = len(scenarios)
    n = len(tickers)
    
    # Create model
    model = pyo.ConcreteModel()
    
    # Decision variables
    model.w = pyo.Var(range(n), domain=pyo.NonNegativeReals)  # Portfolio weights
    model.gamma = pyo.Var(domain=pyo.Reals)  # CVaR threshold
    
    # Helper variables for loss overage
    model.z = pyo.Var(range(S), domain=pyo.NonNegativeReals)
    
    # Objective: minimize CVaR
    coeff = 1.0 / ((1 - alpha) * S)
    model.obj = pyo.Objective(
        expr=model.gamma + coeff * sum(model.z[s] for s in range(S)),
        sense=pyo.minimize
    )
    
    # Constraint 1: Budget (weights sum to 1)
    model.budget = pyo.Constraint(
        expr=sum(model.w[i] for i in range(n)) == 1
    )
    
    # Constraint 2: Target return lower bound
    model.target_lower = pyo.Constraint(
        expr=sum(model.w[i] * expected_returns[i] for i in range(n)) >= target_return - tolerance
    )
    
    # Constraint 3: Target return upper bound
    model.target_upper = pyo.Constraint(
        expr=sum(model.w[i] * expected_returns[i] for i in range(n)) <= target_return + tolerance
    )
    
    # Constraint 4: Loss overage >= 0 (enforced via variable domain, but listed for clarity)
    # z_s >= 0 is already enforced by NonNegativeReals domain
    
    # Constraint 5: Loss constraint for each scenario
    # Loss = 1 - sum(w_t * cumulative_returns[s][t])
    # z_s >= loss - gamma
    model.loss_constraint = pyo.ConstraintList()
    scenario_keys = list(scenarios.keys())
    for s_idx, s_key in enumerate(scenario_keys):
        loss_expr = 1 - sum(model.w[i] * scenarios[s_key][tickers[i]] for i in range(n))
        model.loss_constraint.add(model.z[s_idx] >= loss_expr - model.gamma)
    
    # Create solver
    solver = pyo.SolverFactory("highs")
    solver.options['time_limit'] = time_limit
    
    # Apply user-provided solver options
    if solver_options:
        for key, value in solver_options.items():
            solver.options[key] = value
    
    # Solve
    result = solver.solve(model)
    
    # Check for timeout
    if result.solver.termination_condition == pyo.TerminationCondition.maxTimeLimit:
        print(f"WARNING: Solver timeout at target_return={target_return}")
    
    # Extract results
    weights = {tickers[i]: pyo.value(model.w[i]) for i in range(n)}
    gamma_value = pyo.value(model.gamma)
    cvar_objective = pyo.value(model.obj)
    
    return weights, gamma_value, cvar_objective, result

In [ ]:
# Example: Solve CVaR optimization at 95% confidence level, targeting expected return of 1.01
alpha = 0.95
target_return = 1.3  # 1% expected return

weights, gamma, cvar_obj, result = solve_cvar(
    alpha=alpha,
    scenarios=scenarios,
    tickers=tickers,
    expected_returns=exp_ret,
    target_return=target_return,
    tolerance=0.1,
    time_limit=10
)

print(f"Optimization Result: {result.solver.termination_condition}")
print(f"\nCVaR Objective Value: {cvar_obj:.6f}")
print(f"Gamma (CVaR Threshold): {gamma:.6f}")
print(f"\nPortfolio Weights (top 10):") 
sorted_weights = sorted(weights.items(), key=lambda x: x[1], reverse=True)
for ticker, weight in sorted_weights[:10]:
    print(f"  {ticker}: {weight:.4f}")

NoFeasibleSolutionError: A feasible solution was not found, so no solution can be loaded. Please set opt.config.load_solutions=False and check results.solution_status and results.incumbent_objective before loading a solution.